# LO benchmark (linear optimization) v6

Runs prompts from `data/lp/benchmark.csv`: **9 vignettes × 4 variants = 36 prompts**.

JSON solve: `{"solution": {...}, "cost": <number>}`. Score = keyed `cost` within **1%**.

**Kaggle:** Secrets → `GITHUB_TOKEN` (repo scope, ON). Internet ON. Run setup first.

**Publish:** set `MODEL`, run setup → dry-run (optional) → publish cell → **Build task** / `%choose`.

**Outputs:** `lp_merged_results.csv` and `lp_rate_score_pivot.csv` under `/kaggle/working/`.

In [ ]:
import io
import shutil
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

GITHUB_OWNER = "FrancisGanong-N"
GITHUB_REPO = "sceptical_llms"
GITHUB_BRANCH = "master"
KAGGLE_REPO_DIR = Path("/kaggle/working") / GITHUB_REPO
FORCE_REPO_REFRESH = True
DEBUG_MAX_PROMPTS = None
MAX_OUTPUT_TOKENS = 1024
MODEL = "anthropic/claude-opus-5@default"
# MODEL = "openai/gpt-5.6-sol"


def download_repo_from_github() -> Path:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    url = (
        f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
        f"/zipball/{GITHUB_BRANCH}"
    )
    request = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "kaggle-sceptical-llms-lo-benchmark",
        },
    )

    staging = Path("/kaggle/working") / "_repo_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(staging)
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"GitHub download failed ({exc.code}) for "
            f"github.com/{GITHUB_OWNER}/{GITHUB_REPO}@{GITHUB_BRANCH}: {body[:300]}"
        ) from exc

    extracted = next(p for p in staging.iterdir() if p.is_dir())
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    shutil.copytree(extracted, KAGGLE_REPO_DIR)
    shutil.rmtree(staging)
    return KAGGLE_REPO_DIR


def bootstrap_repo() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "benchmarks" / "lp_rate_tasks.py").is_file():
            return candidate

    if Path("/kaggle/working").is_dir():
        tasks = KAGGLE_REPO_DIR / "benchmarks" / "lp_rate_tasks.py"
        if not FORCE_REPO_REFRESH and tasks.is_file():
            return KAGGLE_REPO_DIR
        return download_repo_from_github()

    raise RuntimeError(
        "Could not find sceptical-llms (need benchmarks/lp_rate_tasks.py). "
        "Run from the repo, or on Kaggle set GITHUB_TOKEN and Internet ON."
    )


ROOT = bootstrap_repo()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for module_name in list(sys.modules):
    if module_name == "benchmarks" or module_name.startswith("benchmarks."):
        del sys.modules[module_name]

print("Repo root:", ROOT)
print("MODEL:", MODEL)
print("DEBUG_MAX_PROMPTS:", DEBUG_MAX_PROMPTS)
print("MAX_OUTPUT_TOKENS:", MAX_OUTPUT_TOKENS)

In [ ]:
import pandas as pd
import kaggle_benchmarks as kbench
from IPython.display import FileLink, display

from benchmarks.lp_rate_tasks import evaluate_lp_rate_benchmark

(
    runs,
    score,
    merged_path,
    pivot_path,
    pivot,
    naive_rate,
    variant_scores,
) = evaluate_lp_rate_benchmark(
    kbench.llms[MODEL],
    max_prompts=DEBUG_MAX_PROMPTS,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    n_jobs=1,
)

print("Model:", MODEL)
print("Overall keyed accuracy:", f"{score.accuracy:.1%}")
print("JSON solve accuracy:", f"{variant_scores['json']:.1%}")
print("Needs tacit accuracy:", f"{variant_scores['needs_tacit_constraint']:.1%}")
print("Detects violation accuracy:", f"{variant_scores['detects_tacit_violation']:.1%}")
print("Naive LP confusion rate:", f"{naive_rate:.1%}")
print("Parse rate:", f"{score.parse_rate:.1%}")
print("Merged results:", merged_path)
print("Score pivot CSV:", pivot_path)

merged_df = pd.read_csv(merged_path)
resp_col = "llm_response" if "llm_response" in merged_df.columns else "response"
empty = int(merged_df[resp_col].astype(str).str.strip().eq("").sum())
print(f"Rows: {len(merged_df)}  |  empty {resp_col}: {empty}")

display(pivot)
display(
    FileLink(merged_path.name, result_html_prefix="Download merged: "),
    FileLink(pivot_path.name, result_html_prefix="Download pivot: "),
)
merged_df.head()

## Publish task

Creates the task `.run.json` with **`MODEL`** from setup (not `kbench.llm`), then `%choose`.
After Build, download runs locally:

```powershell
python scripts/export_lo_kaggle_results.py --download --force-download
```

In [ ]:
import kaggle_benchmarks as kbench
import kaggle_benchmarks.ui.ipython_magics  # registers %choose

from benchmarks.lp_rate_tasks import lo_normative_accuracy_6

print("Publishing with:", MODEL)
run = lo_normative_accuracy_6.run(kbench.llms[MODEL])
print("Task score:", run.result)
print("Task passed:", run.passed)

%choose lo_normative_accuracy_6